In [11]:
import re
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

In [2]:
## Load data from pickled files
df = pd.read_pickle("pos_sent.pkl")
df.head()

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,CMT_SENT,KEY_PHRASES
1,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this I didnt buy ...,8.5,792,0.9751,"[consider somewhat mid im, also strikingly wel..."
2,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318,0.9096,"[ive bought another copy, favorite legacy game..."
3,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite coop game,8.5,23,0.4588,[favorite coop game]
4,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131,0.5106,"[must buy legacy game, cleanest legacy games, ..."
5,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,First complete play through was awesome played...,8.5,97,0.9186,"[first complete play, still great, campaign 2,..."


In [6]:
## Remove common noice in natural language
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
    
## Create list of documents (comments)
documents = np.array(df['COMMENT'])

cleaned_docs = np.vectorize(clean_text)(documents)

In [7]:
cleaned_docs

array(['i was favorably surprised by this i didnt buy it when it first came out because i thought it would be gimmicky change the board as you play but actually it is fantastic now i see what all the hype was about it takes pandemic which i consider somewhat mid im not super into vanilla cooperative games and turns it into a roleplaying like experience with twists and turns in the story line its also strikingly well balanced and well designed even with all of the changes to the rules as the game progresses your abilities scale with the difficulties quite well my wife and i had a blast slight knock to the rating because we probably wont play through it a second time but for a number of months it was the only thing we played thanks to my students for getting it for us',
       'my favorite in the pandemic legacy series and frankly my favorite legacy game period this game was full of so many twists turns and surprises while there is a main storyline that plays out the areascountries impac

In [92]:
custom_stopwords = list(ENGLISH_STOP_WORDS.union({
    "i", "favorite", "in", "game", "great", "good", "fun", "like", "really", "best", "ive", "games",
    "love", "absolutely", "just", "playing", "players", "gloomhaven", "pandemic", "played", "times",
    "far", "legacy", "coop", "amazing", "simply", "legacy", "better", "lot", "dont", "new", "experience",
    "awesome", "gaming", "fantastic", "season", "need", "want", "wait", "people", "friends", "different",
    "player", "excellent", "based", "rating", "perfect", "close", "introduction", "pretty", "set", "im",
    "spirits", "way", "group", "takes", "bit", "handed", "enjoy", "exclusively", "wife", "wish", "copy",
    "willing", "week", "worth", "table", "second", "gets", "having", "probably", "experiences", "recommend",
    "greatest", "truly", "enjoyable", "got", "bought", "usually", "able", "highly", "dd", "favourite", "rpg",
    "recommended", "date", "gamers", "blast", "getting", "think", "id", "family", "buy", "hard", "regular",
    "regularly", "works", "chance", "rated", "wanting", "difficult", "took", "know", "started", "right", "did",
    "excited", "wanted", "day", "waiting", "sure", "days", "year", "years", "joy", "higher", "review", "cards", 
    "board", "plays", "feel", "play", "true", "brain"
}))

In [ ]:
## Create TfIdf vectorizer and fit_transform it to the comments
vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=10,
    stop_words=custom_stopwords,
    ngram_range=(1,3)
)
TfIdf_matrix = vectorizer.fit_transform(cleaned_docs)

## Create the NMF Model for topic analysis
model = NMF(n_components=3, random_state=42)
W = model.fit_transform(TfIdf_matrix)
H = model.components_

In [94]:
## Get the top keywords in each topic
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic #{topic_idx + 1}: {' | '.join(top_words)}")

Topic #1: solo | multiplayer | solo solo | solo mode | mode | solo multiplayer | time solo | solo time | theme | replayability
Topic #2: campaign | story | scenarios | rules | mechanics | characters | theme | gameplay | card | scenario
Topic #3: time | long | long time | setup | setup time | time time | time solo | hours | solo time | time long


# Topics
- Multimodal games with emphasis on solo mode
- Mechanics and thematic element
- Setup and play time

# Strategic Questions

| Question                                                      | What to Do                                                  |
| ------------------------------------------------------------- | ----------------------------------------------------------- |
| Which features do happy players mention most often?           | Count how often each topic appears in **positive** reviews. |
| Which games score highest in each topic?                      | Average topic weights per game in positive reviews.         |
| Are some topics more common in positive vs. negative reviews? | Compare topic distribution across sentiment labels.         |
